# Forelesning 6: Numerisk derivasjon

<div class="bilde-hoyre" style="width:50%">

![Fjell](bilder/mountain.jpg)

</div>

I dag skal vi

- derivere numerisk med ulike tilnærminger
- undersøke hvorfor et mindre steg ikke alltid gir et bedre svar
- derivere eksperimentelle data og finne ekvivalenspunktet i en titrering
- se at lyd og bilder også er tall som kan deriveres

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

<div class="diskuter">

**Diskuter:** I Newtons metode trengte vi den deriverte $f'(x)$. Hva gjør vi hvis funksjonen er vanskelig å derivere, eller hvis vi bare har måledata og ingen formel?

</div>

## Del 1: Fra grenseverdi til differanse

Den deriverte beskriver endring. I kjemi skriver vi den ofte som $\dfrac{dc}{dt}$ (reaksjonsfart) eller $\dfrac{d\mathrm{pH}}{dV}$ (titrerkurve).

<div class="viktig">

Den deriverte er definert som en grenseverdi:

$$f'(x)=\lim_{h\rightarrow 0}\frac{f(x+h)-f(x)}{h}$$

En datamaskin kan ikke bruke en uendelig liten $h$. Vi bruker derfor en liten, endelig steglengde. Det kalles **framoverdifferansen**:

$$f'(x)\approx\frac{f(x+h)-f(x)}{h}$$

</div>

<div class="oppgave">

**Prøv selv:** Skriv en funksjon `forward(f, x, h)` som returnerer framoverdifferansen. Test den på $g(x) = x^2 - 4x + 5$ i $x = 1$, og kontroller svaret med analytisk derivasjon.

</div>

In [ ]:
def g(x):
    return ??

## Del 2: Er mindre steg alltid bedre?

Vi deriverer $\sin x$ i $x = 1$ og sammenlikner med den eksakte verdien $\cos 1$ for $h$ fra $10^{-1}$ til $10^{-16}$.

<div class="diskuter">

**Tipp først:** Hvordan tror du feilen endrer seg når $h$ blir mindre? Fyll inn linja som beregner den absolutte feilen, og kjør koden.

</div>

In [ ]:
h_values = np.logspace(-1, -16, 16)
errors = []

for h in h_values:
    error = ??
    errors.append(error)

plt.loglog(h_values, errors, "o-")
plt.xlabel("h")
plt.ylabel("Absolute error")
plt.show()

<div class="viktig">

To feilkilder konkurrerer:

- **Tilnærmingsfeilen** blir mindre når $h$ blir mindre, fordi sekanten nærmer seg tangenten.
- **Avrundingsfeilen** blir større når $h$ blir svært liten. Vi trekker fra to nesten like flyttall og deler på et svært lite tall.

For framoverdifferansen er $h \approx 10^{-8}$ ofte et godt kompromiss, men den beste verdien avhenger av funksjonen og tallskalaen.

</div>

## Del 3: Bakover- og sentraldifferanse

<div class="bilde-hoyre" style="width:48%">

![Framover-, bakover- og sentraldifferanse](bilder/differanser.png)

</div>

Vi kan også bruke punktet **bak** $x$, eller punkter på **begge sider**.

<div class="viktig">

**Bakover:** $\displaystyle f'(x)\approx\frac{f(x)-f(x-h)}{h}$

**Sentral:** $\displaystyle f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}$

Sentraldifferansen gir vanligvis en bedre tilnærming for samme $h$.

</div>

<div class="oppgave">

**Prøv selv:** Skriv `central(f, x, h)`. Legg den deretter til i feilanalysen, og plott feilen for begge metodene i samme figur. Hvilken metode gir minst feil? Ved hvilken $h$?

</div>

In [ ]:
def central(f, x, h=1e-5):
    return ??

errors_central = []
for h in h_values:
    errors_central.append(??)

plt.loglog(h_values, errors, "o-", label="Forward")
plt.loglog(h_values, errors_central, "s-", label="Central")
plt.xlabel("h")
plt.ylabel("Absolute error")
plt.legend()
plt.show()

<div class="diskuter">

**Diskuter:** Med $h = 10^{-2}$ er feilen omtrent $4 \cdot 10^{-3}$ for framoverdifferansen og $9 \cdot 10^{-6}$ for sentraldifferansen. Hvorfor er sentraldifferansen så mye bedre? Se på figuren med sekantene.

</div>

## Del 4: Derivasjon av måledata

Med måledata har vi ingen formel. Målepunktene bestemmer steglengden, og den er ofte ujevn. Her er en titrering av eddiksyre med 0,10 M NaOH. Filbanen `../datafiler/` viser til mappa med datafilene.

<div class="diskuter">

**Diskuter:** Hvor er ekvivalenspunktet? Hva kjennetegner $\dfrac{d\mathrm{pH}}{dV}$ der?

</div>

<div class="oppgave">

**Finn feilen:** Programmet nedenfor finner ekvivalenspunktet med `np.diff`, som gir differansene mellom nabopunkter. Men stigningene plasseres på feil x-verdier. Hva er galt, og hvor bør hver stigning plasseres?

</div>

In [ ]:
dpH_dV = np.diff(pH) / np.diff(volume)
volume_der = volume[:-1]

V_eq = volume_der[np.argmax(dpH_dV)]
print(f"Equivalence point: {V_eq:.2f} mL")

plt.plot(volume_der, dpH_dV, "o-")
plt.xlabel("Volume of 0.10 M NaOH (mL)")
plt.ylabel("ΔpH/ΔV (1/mL)")
plt.show()

<div class="viktig">

$\dfrac{\Delta y}{\Delta x} = \dfrac{y_{i+1}-y_i}{x_{i+1}-x_i}$ er den **gjennomsnittlige stigningen over intervallet**, og den hører derfor hjemme **midt i intervallet**:

$$x_{\mathrm{midt}}=\frac{x_i+x_{i+1}}{2}$$

</div>

<div class="diskuter">

**Diskuter:** Den største stigningen ligger mellom 33,52 og 33,56 mL. Kan vi rapportere ekvivalenspunktet som 33,54 mL? Hva begrenser presisjonen?

</div>

### `np.gradient`
Når vi har forstått differansene, kan NumPy gjøre jobben. `np.gradient(y, x)` bruker sentraldifferanser i de indre punktene og tåler ujevn avstand mellom x-verdiene.

## Del 5: Alt er tall

Derivasjon handler om **forskjeller**: Hvor mye endrer verdien seg fra ett punkt til det neste? For en datamaskin er lyd og bilder også bare tall. Da kan vi derivere dem på samme måte som en titrerkurve.

<div class="viktig">

- **Lyd** er lufttrykket målt 44 100 ganger i sekundet. Seks og et halvt sekund med lyd er nesten 288 000 tall.
- **Et bilde** er en tabell med piksler. Hvert tall er lysstyrken i ett punkt.

Den deriverte finner vi som før, fra forskjellen mellom nabotall: $\Delta y_i = y_{i+1} - y_i$

</div>

### Derivasjon av lyd

<div class="bilde-hoyre" style="width:40%">

![Utsnitt av lydsignalet](bilder/lyd_original_utsnitt.png)

</div>

`wavfile.read` gir samplingsraten (tall per sekund) og selve lyden som en array. `Audio` spiller av en array.

<div class="diskuter">

**Tipp først:** Hvordan tror du lyden høres ut etter at vi har derivert den?

</div>

In [ ]:
from scipy.io import wavfile
from IPython.display import Audio

<div class="viktig">

<div class="bilde-hoyre" style="width:38%">

![Equalizer med bassreduksjon](bilder/equalizer_bass.png)

</div>

Den deriverte lyden mangler bassen. Lyd som svinger raskt (diskant), gir store forskjeller mellom nabopunkter. Lyd som svinger sakte (bass), gir små forskjeller. Derivasjon virker derfor som en equalizer som skrur ned bassen.

Det er også grunnen til at derivasjon forsterker **målestøy**: Støy varierer raskt fra punkt til punkt.

</div>

### Derivasjon av bilder

Et fargebilde er tre tabeller oppå hverandre: rød, grønn og blå. Gjennomsnittet av de tre gir et gråtonebilde. `np.gradient` på en tabell gir den deriverte i både y- og x-retning.

<div class="oppgave">

**Prøv selv:** Fyll inn linjene som deriverer gråtonebildet og legger sammen størrelsen på endringene i begge retninger.

</div>

In [ ]:
img = plt.imread("../datafiler/kjemibygningen.jpg")
gray = np.mean(img, axis=2)

dy, dx = ??
edges = ??

plt.imshow(edges, cmap="gray", vmax=60)
plt.show()

<div class="diskuter">

**Diskuter:** Hvor blir det deriverte bildet lyst? Hva tilsvarer ekvivalenspunktet i titreringen i et bilde?

</div>

__Oppgaver__:
1. Beregn $f'(2)$ numerisk med sentraldifferansen og kontroller analytisk for $f(x) = e^x$ og $f(x) = \sqrt{\ln x}$.
2. Konsentrasjonen av A i en reaksjon er målt ved `t = [0, 10, 20, 30, 40, 50]` s: `c = [1.00, 0.82, 0.68, 0.56, 0.47, 0.40]` mol/L. Beregn $-\Delta c/\Delta t$ og plott reaksjonsfarten mot midtpunktene i tidsintervallene. Hvordan endrer farten seg med tiden, og hvorfor?
3. En forenklet potensiell energi er $U(r)=r^{-12}-2r^{-6}$. Bruk sentraldifferansen til å finne omtrent hvor $dU/dr=0$. Kontroller ved å plotte $U(r)$. Hva betyr dette punktet kjemisk?
4. Bytt ut den analytiske deriverte i Newtons metode fra forrige forelesning med `central`. Finn pH i 0,010 M eddiksyre med steglengdene $10^{-8}$, $10^{-5}$ og $3 \cdot 10^{-4}$. Hvor mange iterasjoner trengs i hvert tilfelle? Hvorfor må steglengden velges i forhold til størrelsen på $[\mathrm{H_3O^+}]$?
5. Lag et datasett med ujevnt fordelte tidspunkter fra en kjent funksjon. Sammenlikn en beregning som feilaktig antar konstant $\Delta t$ med `np.gradient(y, t)`.
6. Deriver bildet `sommerfugl.jpg` bare i x-retning og bare i y-retning. Plott de to resultatene ved siden av hverandre. Hvilke kanter får du fram i hvert tilfelle, og hvorfor?